# MalImg Family Classification

**Task 3 — Machine Learning with Image Data | Semester I, 2026–2027**

Base để nhóm review, chỉnh cấu hình và chạy thực nghiệm. Dataset MalImg 24 lớp
(loại Yuner.A) đang chờ giảng viên xác nhận. Notebook chưa có kết quả benchmark
hoặc bằng chứng đã chạy `Run all` từ Colab sạch.

- A: dữ liệu, EDA, split; B: đặc trưng, classifier; C: đánh giá, tích hợp.
- Điền tên nhóm, họ tên/MSSV/email, mã môn và giảng viên trước khi nộp.
- Repo: https://github.com/TuanKhai1210/malimg-family-classification
- Xem `docs/review_and_train.md` và `reports/review_checklist.md`.

Trong Colab chọn **Runtime → Change runtime type → GPU** trước khi chạy pretrained
features. Không cần mount Drive. `prepare` dành cho review; `all` chạy EDA,
trích/lưu/nạp features và train/chọn C bằng validation. Chỉ bật final test sau
khi nhóm khóa phương án. Bản nộp cuối cần chạy toàn bộ trên runtime sạch và có report.


## 1. Chọn chế độ

Chỉnh một cell dưới đây, rồi **Runtime → Run all**. Lần đầu giữ `STAGE='prepare'`
để kiểm kê dữ liệu, split và EDA. Sau review đổi sang `STAGE='all'` để train.
`FINAL_TEST=True` mở đánh giá test sau chọn bằng validation; kết quả được khóa,
không tự chạy lại khi thay cấu hình. Dữ liệu ~1.17 GB, weights sẽ tải từ PyTorch.


In [ ]:
STAGE = 'prepare'  # prepare | eda | extract | train | final | all
FINAL_TEST = False  # dùng với all sau khi khóa protocol; final là bước riêng
INSTALL_DEPENDENCIES = True
REPO_URL = 'https://github.com/TuanKhai1210/malimg-family-classification.git'
REPO_REF = 'feat/reviewable-core-pipeline'
CONFIG_FILE = 'configs/project.json'
ARCHIVE_PATH = None  # Colab tự tải link công khai; local có thể dùng ZIP đã kiểm


## 2. Chuẩn bị code và thư viện

Cell này dùng repo hiện tại nếu đã có, hoặc clone đúng nhánh review. Ghi lại commit
và môi trường thật cho mỗi lần chạy. Giữ sẵn PyTorch của Colab nếu đáp ứng yêu cầu;
nếu cài/thay thư viện trong một runtime đã dùng trước đó, hãy restart và Run all.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

candidates = [Path.cwd(), Path.cwd() / 'malimg-family-classification',
              Path('/content/malimg-family-classification')]
repo_root = next((path for path in candidates if (path / 'modules/pipeline.py').is_file()), None)
if repo_root is None:
    parent = Path('/content') if Path('/content').is_dir() else Path.cwd()
    repo_root = parent / 'malimg-family-classification'
    if repo_root.exists():
        raise RuntimeError('Repo folder already exists but lacks pipeline code. Use a clean runtime or inspect it.')
    subprocess.run(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(repo_root)], check=True)
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(repo_root / 'requirements.txt')], check=True)
sys.path.insert(0, str(repo_root))
print('Repository:', repo_root)
if (repo_root / '.git').exists():
    print('Commit:', subprocess.check_output(['git', '-C', str(repo_root), 'rev-parse', 'HEAD'], text=True).strip())
elif (repo_root / 'PACKAGE.json').is_file():
    print('Packaged source commit:', json.loads((repo_root / 'PACKAGE.json').read_text(encoding='utf-8')).get('source_commit'))
else:
    print('Source uploaded without Git metadata; record its origin before benchmark use.')
subprocess.run([sys.executable, str(repo_root / 'scripts/check_environment.py')], check=True)


## 3. Cấu hình chung

Core: E2 ResNet50+LR, E3 ResNet50+SVM, E4 ConvNeXt-Tiny+SVM. Chỉ train được fit
scaler/PCA. C được chọn bằng Macro-F1 của đại diện nhóm pixel trên validation.
Tất cả trial, cảnh báo hội tụ, config, sample IDs và provenance được lưu.

Nguồn ảnh công khai (download trực tiếp, không mount cloud drive):
https://www.dropbox.com/scl/fi/wdb6omeiu2lg796qvt9l7/malimg_dataset.zip?rlkey=63q2xqmtlm66gilf6idd2c9k7&dl=1

Checksum bản đã kiểm: `9766ae9f1daa520e367fb486ca94728fe1485c0f5cb8314c312d77089a1fe9ec`.


In [ ]:
from modules.config import load_config, validate_config

config = load_config(repo_root / CONFIG_FILE)
# Chỉnh các lựa chọn tại đây trước lần validation đầu tiên:
# config['features']['batch_size'] = 16
# config['features']['device'] = 'cuda'
# config['features']['resize_policy'] = 'pad'
# config['training']['pca_components'] = 0.95
validate_config(config)
print(json.dumps(config, ensure_ascii=False, indent=2))


## 4. Chạy pipeline

Prepare kiểm archive, chỉ giải nén ảnh, tạo manifest/label map/split theo nhóm pixel,
kiểm leakage và dựng EDA. Feature cache chỉ tái dùng khi provenance và hashes khớp.
Test features có thể được trích sẵn; validation không nạp mảng test hoặc chấm test.

`all` giữ test kín khi `FINAL_TEST=False`. Sau khi nhóm khóa phương án,
chọn `STAGE='final'` trong cùng runtime, hoặc `all` + `FINAL_TEST=True` cho lần
kiểm tra bản nộp từ runtime sạch. Không bỏ lock để tìm cấu hình theo test.


In [ ]:
from modules.pipeline import run_pipeline

if FINAL_TEST and STAGE != 'all':
    raise ValueError('Use FINAL_TEST with all, or choose the explicit final stage.')
run = run_pipeline(repo_root, config, stage=STAGE,
                   archive_path=ARCHIVE_PATH, final_test=FINAL_TEST)
print(json.dumps(run, ensure_ascii=False, indent=2, default=str))


## 5. Xem EDA và kết quả thật

Hai đơn vị đánh giá được trình bày riêng: mỗi nhóm pixel một đại diện và toàn bộ
file. Macro-F1 là metric chính. Các bảng sau chỉ hiện khi đã có output thật;
không dùng kết quả synthetic smoke tests trong báo cáo MalImg.


In [ ]:
import pandas as pd
from IPython.display import display, Image as DisplayImage

results_root = repo_root / config.get('results_dir', 'results')
summary_path = Path(run['dataset']['summary_path'])
print(summary_path.read_text(encoding='utf-8'))
for path in sorted((results_root / 'eda').glob('*.png')):
    print(path.name)
    display(DisplayImage(filename=str(path)))
comparison = results_root / 'comparison.csv'
if comparison.is_file():
    display(pd.read_csv(comparison))
else:
    print('Chưa chạy classifier; chưa có bảng benchmark.')


## 6. Lưu và đóng gói

Colab runtime có thể mất dữ liệu khi ngắt kết nối. Tải ZIP draft và artifacts về
máy sau khi chạy; lần sạch tiếp theo có thể tái tính từ nguồn công khai. Draft
chưa phải gói nộp hoàn chỉnh. Báo cáo PDF, thông tin môn/nhóm, đóng góp thật,
giảng viên duyệt và kiểm Run all vẫn do nhóm hoàn thiện.

Model `joblib` chỉ được nạp từ output do chính nhóm tạo và đã kiểm hash;
không nạp pickle/joblib tải từ nguồn chưa tin cậy.


In [ ]:
CREATE_DRAFT_ZIP = False  # đổi True khi muốn tải code và artifacts về máy
if CREATE_DRAFT_ZIP:
    from modules.submission import build_submission
    package = build_submission(repo_root, repo_root / 'dist/malimg_review_base.zip', draft=True)
    print('Draft ZIP:', package)
    try:
        from google.colab import files
    except ImportError:
        print('Local runtime: mở file ZIP ở đường dẫn trên.')
    else:
        files.download(str(package))


## 7. Checklist và tham khảo

- Review dữ liệu, chính sách Yuner.A và split; xác nhận phạm vi với giảng viên.
- Kiểm feature/image/label alignment; chốt weights, resize, C grid và train unit.
- Chạy validation, đọc warnings; khóa lựa chọn rồi đánh giá test.
- Điền báo cáo từ artifacts thật và ghi nhận phần nhóm đã review/chỉnh sửa.
- Thành viên thứ hai kiểm notebook từ Colab sạch; lưu log/evidence.
- Xuất `reports/final_report.pdf`, hoàn thiện README/thành viên và tạo ZIP cuối.

Nataraj et al., [Malware Images: Visualization and Automatic Classification](https://doi.org/10.1145/2016904.2016908), VizSec 2011.
[Torchvision ResNet50](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet50.html),
[ConvNeXt-Tiny](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.convnext_tiny.html),
[scikit-learn model persistence](https://scikit-learn.org/stable/model_persistence.html).
